# 00 — Verify Connection & Create WAF-Tagged Namespace

> This is the *governance skeleton* for the whole demo. Every later notebook
> inherits the catalog/schema comments and cost-center tags you plant here.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Data Governance** | Three medallion schemas with descriptive `COMMENT ON`s — shows up in Catalog Explorer, Discover, Genie, AI/BI. Good comments are literally worth tens of % of accuracy on Genie answers. |
| **Cost Optimization** | Catalog-level tags (`cost_center`, `env`, `removeAfter`) propagate into `system.billing.usage.custom_tags` — every DBU downstream is automatically attributable to this demo. Show this slide to finance. |
| **Operational Excellence** | `.env`-driven config + `DatabricksSession.builder.serverless()` — no cluster sizing, no hardcoded workspace URL, demo runs unchanged on any SA laptop. |
| **Reliability** | A UC Volume for checkpoint state means Auto Loader (notebook 02) can restart from exactly where it left off even if the laptop crashes mid-run. |

The three schemas:

- `{UC_SCHEMA}_bronze` — raw VARIANT JSON from the MLB GUMBO API
- `{UC_SCHEMA}_silver` — typed, deduplicated tables with PK/FK + CHECK constraints
- `{UC_SCHEMA}_gold`   — star schema powering dashboards, Genie, and ML


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
print("Spark version:", spark.version)
spark.sql("SELECT current_user(), current_timestamp()").show(truncate=False)


Spark version: 4.1.0


+------------------------------+--------------------------+
|current_user()                |current_timestamp()       |
+------------------------------+--------------------------+
|alexander.booth@databricks.com|2026-04-21 18:41:07.574483|
+------------------------------+--------------------------+



In [2]:
UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")

BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

COST_CENTER   = os.getenv("WAF_COST_CENTER",  "field_engineering")
ENV_TAG       = os.getenv("WAF_ENV",           "demo")
DATA_OWNER    = os.getenv("WAF_DATA_OWNER",    "mlb_analytics_team")
REMOVE_AFTER  = os.getenv("WAF_REMOVE_AFTER",  "20281231")

print(f"Target: {UC_CATALOG}.{{{BRONZE_SCHEMA}, {SILVER_SCHEMA}, {GOLD_SCHEMA}}}")
print(f"WAF tags: cost_center={COST_CENTER}, env={ENV_TAG}, owner={DATA_OWNER}, removeAfter={REMOVE_AFTER}")


Target: alexander_booth.{mlb_gumbo_waf_bronze, mlb_gumbo_waf_silver, mlb_gumbo_waf_gold}
WAF tags: cost_center=field_engineering, env=demo, owner=mlb_analytics_team, removeAfter=20281231


## Tag the catalog — chargeback-as-code

> **WAF — Cost Optimization.** Catalog tags flow to every asset inside and to
> `system.billing.usage.custom_tags`. This is the foundation for the tag-based
> chargeback query we'll build in notebook 07. If a customer only takes one
> thing from this demo, it should be: *tag your catalogs on day zero*.


In [3]:
def set_tag(subject_sql, key, value):
    """Apply one tag; warn but continue if the workspace has a tag policy that rejects it."""
    try:
        spark.sql(f"{subject_sql} SET TAGS ('{key}' = '{value}')")
        return True
    except Exception as e:
        msg = str(e)
        if "INVALID_PARAMETER_VALUE" in msg or "not an allowed value" in msg or "tag policy" in msg.lower():
            print(f"    skipped tag {key}={value} (governed tag policy)")
        else:
            print(f"    skipped tag {key}={value} ({msg[:120]})")
        return False


# Catalog creation is a no-op if the catalog already exists. If you do not
# have CREATE CATALOG privileges, skip this cell and ensure {UC_CATALOG} exists.
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {UC_CATALOG} COMMENT 'Alexander Booth demo catalog'")
    print(f"Catalog ready: {UC_CATALOG}")
    for k, v in [("cost_center", COST_CENTER), ("env", ENV_TAG),
                 ("data_owner", DATA_OWNER), ("removeAfter", REMOVE_AFTER)]:
        set_tag(f"ALTER CATALOG {UC_CATALOG}", k, v)
except Exception as e:
    print(f"Catalog step skipped: {e}")


Catalog ready: alexander_booth


    skipped tag cost_center=field_engineering (governed tag policy)


## Create medallion schemas + landing volume

> **WAF — Data Governance.** Every schema gets a human-readable `COMMENT`
> describing *purpose*, *shape*, and *grain*. This is literally the first
> thing Catalog Explorer shows when someone clicks a schema, and Genie uses
> it when generating answers. Short, specific comments > long prose.


In [4]:
schema_specs = [
    (BRONZE_SCHEMA, "Bronze — raw MLB GUMBO JSON landed via Auto Loader, stored as VARIANT with source-file metadata."),
    (SILVER_SCHEMA, "Silver — typed + deduplicated game and pitch tables with RELY PK/FK constraints and CHECK data quality rules."),
    (GOLD_SCHEMA,   "Gold — analytics-ready star schema (dims + facts + MVs) powering the pitch analytics dashboard, Genie, and ML."),
]

for schema, comment in schema_specs:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema} COMMENT '{comment}'")
    print(f"Schema ready: {UC_CATALOG}.{schema}")
    for k, v in [("cost_center", COST_CENTER), ("env", ENV_TAG), ("data_owner", DATA_OWNER)]:
        set_tag(f"ALTER SCHEMA {UC_CATALOG}.{schema}", k, v)

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.raw_gumbo
    COMMENT 'Landing zone for raw MLB GUMBO JSON files and Auto Loader checkpoints.'
""")
print(f"  Volume ready: {UC_CATALOG}.{BRONZE_SCHEMA}.raw_gumbo")


Schema ready: alexander_booth.mlb_gumbo_waf_bronze


    skipped tag cost_center=field_engineering (governed tag policy)


Schema ready: alexander_booth.mlb_gumbo_waf_silver


    skipped tag cost_center=field_engineering (governed tag policy)


Schema ready: alexander_booth.mlb_gumbo_waf_gold


    skipped tag cost_center=field_engineering (governed tag policy)


  Volume ready: alexander_booth.mlb_gumbo_waf_bronze.raw_gumbo


In [5]:
spark.sql(f"DESCRIBE SCHEMA EXTENDED {UC_CATALOG}.{BRONZE_SCHEMA}").show(truncate=False)


+-------------------------+------------------------------------------------------------------------------------------------+
|database_description_item|database_description_value                                                                      |
+-------------------------+------------------------------------------------------------------------------------------------+
|Catalog Name             |alexander_booth                                                                                 |
|Namespace Name           |mlb_gumbo_waf_bronze                                                                            |
|Comment                  |Bronze — raw MLB GUMBO JSON landed via Auto Loader, stored as VARIANT with source-file metadata.|
|Location                 |                                                                                                |
|Owner                    |alexander.booth@databricks.com                                                                  |


## Optional: Reset (keep the landing Volume)

The reset cell below drops every Delta table in the three demo schemas **but
preserves the `raw_gumbo` Volume** — so you can re-demo the pipeline (02 → 11)
without re-hitting the MLB API for ~2,500 JSON files.

If you *also* want to wipe the Volume (full from-scratch run starting at 01),
run the second cell instead.


In [ ]:
# ── Reset everything EXCEPT the landing Volume ──────────────────────────────
# Drops every Delta table in bronze/silver/gold so the pipeline can be
# re-run, but leaves /Volumes/{cat}/{bronze}/raw_gumbo intact so you don't
# need to re-download any JSON. Also clears the Auto Loader checkpoint so
# notebook 02 will re-process every file.
#
# Uncomment and run when you want to reset.

# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
#
# for schema in [GOLD_SCHEMA, SILVER_SCHEMA, BRONZE_SCHEMA]:
#     tables = spark.sql(f"SHOW TABLES IN {UC_CATALOG}.{schema}").collect()
#     for row in tables:
#         full = f"{UC_CATALOG}.{schema}.{row['tableName']}"
#         spark.sql(f"DROP TABLE IF EXISTS {full}")
#         print(f"  dropped {full}")
#
# # Wipe Auto Loader checkpoint + schema dirs so notebook 02 re-ingests cleanly
# def _rm_rf(path):
#     try:
#         items = list(w.files.list_directory_contents(path))
#     except Exception:
#         try: w.files.delete(path)
#         except Exception: pass
#         return
#     for it in items:
#         if it.is_directory: _rm_rf(it.path)
#         else:
#             try: w.files.delete(it.path)
#             except Exception: pass
#     try: w.files.delete_directory(path)
#     except Exception: pass
#
# volume_root = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/raw_gumbo"
# _rm_rf(f"{volume_root}/_checkpoints")
# _rm_rf(f"{volume_root}/_schemas")
# print("Checkpoint + schema state cleared. The raw JSON landing files are preserved.")


In [ ]:
# ── Nuclear reset: drop the schemas AND wipe the landing Volume ─────────────
# Uncomment only if you want to re-download every JSON file from the MLB API.

# for schema in [GOLD_SCHEMA, SILVER_SCHEMA, BRONZE_SCHEMA]:
#     spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{schema} CASCADE")
#     print(f"Dropped schema {UC_CATALOG}.{schema} (CASCADE removes the Volume too)")
